# cancer-recon-apoptosis — RUNG 5: worst-case-safety surfaceome re-audit + the ADDRESSABILITY GAP

Re-audit surface logic-gate recognition across the whole OmniPath/SURFY surfaceome, donor-aware, with worst-DONOR safety + held-out-donor FDR + winner's-curse shrinkage, and report the **per-patient addressability gap** (fraction with NO safe gate, by cancer type). *"Most patients have no clean gate"* is the expected first-class **negative**.

**Runtime:** CPU is fine and is what the fetch needs (network/disk-bound — GPU does nothing for it). **If Colab only gives you a GPU runtime, that's OK** — Cell 4b detects it and the scorer (the one GPU-accelerable, RAM-bound step) uses it, and Cell 5 bumps the rigor (more permutations + bigger search) since GPU makes that affordable. Same validated math, just more of it. The fetch is cached on your Drive, so re-runs skip the ~1-hour download.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recon-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — mount Google Drive for a RESUMABLE cache (recommended) + start run log
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    cache_dir = '/content/drive/MyDrive/cancer-recon'
    os.makedirs(cache_dir, exist_ok=True)
    os.environ['LOGICGATE_CACHE'] = f'{cache_dir}/rung5_cache.npz'
    print('[CELL 2] Drive mounted — cached fetch (tumour + 9 tissue tiles) loads instantly; only scoring re-runs.')
except Exception as e:
    os.environ['LOGICGATE_CACHE'] = '/content/cancer-recon-apoptosis/data/logicgate/rung5_cache.npz'
    print('[CELL 2] Drive NOT mounted (', type(e).__name__, ') — caching to /content (LOST on disconnect).')
from scripts.runlog import new_log
RUNLOG = new_log('rung5_addressability', repo_dir='.')
print('[CELL 2] ✓')

In [ ]:
#@title Cell 3 — VALIDATE the rigor layer + the full downstream on synthetic ground truth (CPU, no data)
import sys
from scripts.runlog import run_logged
rc1 = run_logged([sys.executable, '-u', 'scripts/23_heldout_validate.py'], RUNLOG)
rc2 = run_logged([sys.executable, '-u', 'scripts/25_logicgate_data_rung5.py', 'selftest'], RUNLOG)
print(f'[CELL 3] scripts/23 exit={rc1}  scripts/25 selftest exit={rc2}',
      '✓ both validated' if rc1 == 0 and rc2 == 0 else '✗ validation FAILED — stop here')
from IPython.display import Image, display
import os
for p in ['runs/rung5_logicgate/rung5_heldout_validation.png', 'runs/rung5_logicgate/rung5_selftest.png']:
    if os.path.exists(p):
        display(Image(p))

In [ ]:
#@title Cell 4 — install CELLxGENE Census + OmniPath (only needed for the real run)
import sys, importlib.util
from scripts.runlog import run_logged
for pkg, pip_name in [('cellxgene_census', 'cellxgene-census'), ('scanpy', 'scanpy'), ('omnipath', 'omnipath')]:
    if importlib.util.find_spec(pkg) is None:
        run_logged([sys.executable, '-m', 'pip', 'install', '-q', pip_name], RUNLOG, label=f'pip install {pip_name}')
print('[CELL 4] ✓ (if Colab asks to RESTART runtime, do it, then Run all again — the cache makes that cheap)')

In [ ]:
#@title Cell 4b — GPU check + benchmark the bincount the scorer offloads (the RAM-bound hot loop)
# The fetch can't use a GPU (network/disk). The SCORING (boolean gate-firing + per-group bincount over the
# ~1.26M-cell panel) can. This cell confirms CuPy/GPU and benchmarks the exact op the scorer offloads.
import os
try:
    import cupy as cp, numpy as np, time
    ndev = cp.cuda.runtime.getDeviceCount()
    print(f'[GPU] CuPy devices: {ndev}', '(', cp.cuda.runtime.getDeviceProperties(0)['name'].decode(), ')' if ndev else '')
    if ndev:
        N = 5_000_000; g = np.random.randint(0, 50000, N); w = (np.random.random(N) > 0.5).astype(float)
        t = time.time(); np.bincount(g, weights=w, minlength=50000); tc = time.time() - t
        gg, gw = cp.asarray(g), cp.asarray(w); cp.cuda.Stream.null.synchronize()
        t = time.time(); cp.bincount(gg, weights=gw, minlength=50000); cp.cuda.Stream.null.synchronize(); tg = time.time() - t
        print(f'[GPU] bincount(5M cells) CPU {tc*1000:.0f}ms vs GPU {tg*1000:.0f}ms  -> {tc/max(tg,1e-6):.1f}x faster')
        print('[GPU] -> scoring will run on GPU (identical math, validated bit-equivalent); Cell 5 raises the rigor.')
        os.environ['R5_GPU'] = '1'
    else:
        os.environ['R5_GPU'] = '0'; print('[GPU] no device -> scoring uses CPU (the validated path; already fast).')
except Exception as e:
    os.environ['R5_GPU'] = '0'; print('[GPU] CuPy unavailable (', type(e).__name__, ') -> CPU scoring (validated, fine).')

In [ ]:
#@title Cell 5 — REAL run: donor-aware addressability gap (resumes from cache)
#@markdown RUNTIME-SAFE defaults (~15-25 min; finish whether or not the GPU actually engages). The scorer
#@markdown uses the GPU automatically if Cell 4b found one (faster), but the knobs stay modest so a CPU
#@markdown fallback still finishes. ONLY raise MAX_FAMILY / N_PERM if Cell 4b showed a big GPU speedup AND
#@markdown you SEE GPU RAM climb during scoring (else they grind on CPU and blow past the runtime).
K_ACTIVATORS = 80   #@param {type:"integer"}
COV_FLOOR = 0.05    #@param {type:"number"}
GENE_FLOOR = 0.02   #@param {type:"number"}
MAX_FAMILY = 1500   #@param {type:"integer"}
N_PERM = 40         #@param {type:"integer"}
N_BOOT = 40         #@param {type:"integer"}
import os, sys
print('[CELL 5] scoring backend:', 'GPU' if os.environ.get('R5_GPU')=='1' else 'CPU',
      f'| MAX_FAMILY={MAX_FAMILY} N_PERM={N_PERM} N_BOOT={N_BOOT}  (watch the [fdr]/[boot] progress lines)')
os.environ.update(R5_K_ACTIVATORS=str(K_ACTIVATORS), R5_COV_FLOOR=str(COV_FLOOR), R5_GENE_FLOOR=str(GENE_FLOOR),
                  R5_MAX_FAMILY=str(MAX_FAMILY), R5_N_PERM=str(N_PERM), R5_N_BOOT=str(N_BOOT))
rc = run_logged([sys.executable, '-u', 'scripts/25_logicgate_data_rung5.py'], RUNLOG)
print(f'[CELL 5] exit={rc}')
import json
p = 'runs/rung5_logicgate/rung5_addressability.json'
if os.path.exists(p):
    r = json.load(open(p)); a = r.get('addressability', {})
    print('full surfaceome / screened-expressed:', r.get('surfaceome_full'), '->', r.get('surfaceome_screened_expressed'))
    print('cells/patients/genes :', r.get('n_cells'), '/', r.get('n_donors'), '/', r.get('n_genes'))
    print('family size          :', r.get('family_size'), '| SAFE:', r.get('n_safe'), '| survive FDR+shrinkage+GuardB:', r.get('n_survivors'))
    print('ADDRESSABILITY GAP   :', a.get('addressability_gap_overall'), 'worst-case over ALL',
          a.get('n_patients'), 'patients  (', a.get('n_underpowered_patients'), 'under-powered )')
    print('  by cancer type     :', a.get('addressability_gap_by_cancer_type'))
    print('survivors (top)      :', r.get('survivors'))
from IPython.display import Image, display
if os.path.exists('runs/rung5_logicgate/rung5_real.png'):
    display(Image('runs/rung5_logicgate/rung5_real.png'))

In [ ]:
#@title Cell 6 — finalize + download run log AND all figures / result files
import os, glob
from scripts.runlog import finalize
finalize(RUNLOG)
EXTRA = (glob.glob('runs/rung5_logicgate/*.png') + glob.glob('runs/rung5_logicgate/*.json')
         + glob.glob('data/logicgate/surfaceome_genes.txt'))
try:
    from google.colab import files
    for p in sorted(set(EXTRA)):
        if os.path.exists(p):
            print('[download]', p); files.download(p)
except ImportError:
    print('(not in Colab — download skipped)')
except Exception as e:
    print('(download skipped:', type(e).__name__, e, ')')

## What this rung establishes

- Harness validated before the data (Cell 3); the real run is worst-DONOR safe, fail-closed, dropout-guarded, no-multiply; coverage must beat the **family-max decoy FDR** + survive **winner's-curse shrinkage** + **GUARD B** (donor-permutation null) on held-out donors.
- **Deliverable = the per-patient ADDRESSABILITY GAP** (worst-case over ALL patients; under-powered counted as not-addressed). *"Most patients have no clean surface gate"* is the expected, first-class **negative**.

**GPU note:** the GPU only accelerates the RAM-bound scoring (boolean firing + bincount) — bit-identical to CPU (validated). It does **not** change the answer; it lets the GPU run afford a more rigorous search/FDR in the same time. The fetch is network-bound and never uses GPU.

**Ceiling:** transcript-only (mRNA ≠ surface protein; CITE-seq confirms co-positivity); NOT first at combinatorial search — the contribution is the worst-case-safety honesty harness + the addressability gap. A surviving gate is the best NEXT wet-lab experiment, not a cure.